# Project LANTERN: from a BDC filing to a traceable panel

**Audience:** a classmate or grader who knows Python but not business-development companies (BDCs).

**Goal:** trace one real SEC filing through the same loaders and structural rules used by the pipeline, then verify why it is admitted and state the coverage limits honestly.

The notebook reads the real release named in `params.yaml`. It fails clearly if the repository is in fixture mode. It is read-only and is not part of `dvc.yaml`.

**What could go wrong:** run `dvc pull` first and use the repository root (or open this notebook from Jupyter started at the root). If `sec.gov` is unavailable, the live filing comparison is skipped; release-based steps still run.

In [1]:
from pathlib import Path
import json
import re
import sys

import pandas as pd
import requests
from bs4 import BeautifulSoup

root = Path.cwd()
if not (root / 'params.yaml').exists():
    root = root.parent
if not (root / 'params.yaml').exists():
    raise RuntimeError('Start Jupyter from the lantern-lab2b repository root')
sys.path.insert(0, str(root / 'src'))

from lookthrough import clean, coalesce, holdings, load, mapping, paths, reconcile

params = paths.load_params(root)
if params['data']['use_fixture']:
    raise RuntimeError('Set data.use_fixture to false; this lab requires the real release')
release_dir = paths.release_dir(params, root)
if not release_dir.name.startswith(params['release']):
    raise RuntimeError(f'Expected release {params["release"]}, found {release_dir.name}')
print('Release:', release_dir.name)
print('Fixture mode:', params['data']['use_fixture'])

Release: 2026q3_bdc
Fixture mode: False


## Step 1 — What a BDC is and what a schedule of investments looks like

A BDC lends to and invests in smaller companies. Its schedule of investments (SOI) is the table of those positions. The SEC release gives us a flat version of that table, while the filing gives a reader a formatted version.

**Why this matters:** a portfolio number should be traceable to a row a human can inspect. We will use **STELLUS CAPITAL INVESTMENT CORP**, accession `0001104659-26-093435`, which was chosen because it has many multi-row holdings and a structural cleaning decision.

**What could go wrong:** the filing's display label may differ slightly from the XBRL identifier, and `sec.gov` can be unavailable. We compare sampled rows only and do not invent unavailable filing text.

In [2]:
ADSH = '0001104659-26-093435'
PERIOD = '2026-06-30'
TRACE_IDENTIFIER = '2X LLC Investors LP Class A Units'
TRACE_DISPLAY = '2X Investors LP Class A Units'

sub = load.load_sub(release_dir)
tag = load.load_tag(release_dir)
header = load.soi_header(release_dir)
colmap = mapping.map_soi_columns(header, tag)
usecols = mapping.soi_usecols(header, colmap)
if coalesce.SUBSIDIARY_AXIS in header:
    usecols += [coalesce.SUBSIDIARY_AXIS]
soi_release = load.load_soi(release_dir, usecols)
soi = soi_release[soi_release['adsh'].eq(ADSH)].copy()
if soi.empty:
    raise RuntimeError('Traced filing is absent from the real release')
print('Traced filing:', sub.loc[sub.adsh.eq(ADSH), 'name'].iloc[0])
print('Filing rows in release:', len(soi))
canonical_base = mapping.to_canonical(soi, colmap)
canonical_base['subsidiary'] = (
    soi[coalesce.SUBSIDIARY_AXIS].copy()
    if coalesce.SUBSIDIARY_AXIS in soi.columns else pd.NA
)

Traced filing: STELLUS CAPITAL INVESTMENT CORP
Filing rows in release: 2168


In [3]:
inline_url = soi['inlineurl'].dropna().iloc[0]
direct_url = inline_url.replace('https://www.sec.gov/ix?doc=/', 'https://www.sec.gov/')
headers = {'User-Agent': params['sec']['user_agent']}
SEC_AVAILABLE = False
human_row = None

try:
    response = requests.get(direct_url, headers=headers, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    node = soup.find(string=lambda s: s and s.strip() == TRACE_DISPLAY)
    table = node.find_parent('table') if node else None
    if table is None:
        raise LookupError('Display row not found in the filing HTML')
    human_row = table.find_all('tr')[9]
    cells = [
        ' '.join(cell.get_text(' ', strip=True).replace('\u200b', '').split())
        for cell in human_row.find_all(['th', 'td'])
    ]
    SEC_AVAILABLE = True
    print('Human-readable Stellus schedule row:')
    print(' | '.join(part for part in cells if part))
    print('EDGAR:', inline_url)
except Exception as exc:
    print('SEC filing fetch skipped:', type(exc).__name__)
    print('The raw release rows below remain the reproducible evidence.')

Human-readable Stellus schedule row:
2X Investors LP Class A Units | Equity | 6/5/2023 | 43,875 | 176,915 | 1,220,895 | 0.33 | %
EDGAR: https://www.sec.gov/ix?doc=/Archives/edgar/data/1551901/000110465926093435/scm-20260630x10q.htm


In [4]:
raw_example = soi[soi['Investment, Identifier Axis'].eq(TRACE_IDENTIFIER)]
canonical_example = mapping.to_canonical(raw_example, colmap)
columns = ['soi_row', 'ddate', 'identifier', 'fair_value', 'cost', 'shares',
           'principal', 'pct_net_assets', 'investment_type']
display(canonical_example[[c for c in columns if c in canonical_example]].reset_index(drop=True))

,soi_row,ddate,identifier,fair_value,cost,shares,principal,pct_net_assets,investment_type
0,42668,2025-12-31,2X LLC Investors LP Class A Units,1220397.0,176915.0,43875.0,NaN,0.0033,NaN
1,42669,2025-12-31,2X LLC Investors LP Class A Units,NaN,NaN,NaN,NaN,NaN,NaN
2,42670,2026-06-30,2X LLC Investors LP Class A Units,1220895.0,176915.0,43875.0,NaN,0.0033,NaN
3,42671,2026-06-30,2X LLC Investors LP Class A Units,NaN,NaN,NaN,NaN,NaN,NaN


**Expected output:** the filing row and the release rows name the same investment. Current source rows `42670;42671` carry values on 42670 and dimension facts on 42671; rows 42668–42669 are the 2025 comparative copy.

**Why the raw shape surprises people:** XBRL is fact-oriented rather than spreadsheet-oriented. One displayed investment can be represented by several rows.

**What could go wrong:** do not count all four source rows as four investments. Period and identifier define the candidate, and later coalescing combines fields.

## Step 2 — The data: SUB, TAG, NUM and SOI

**Goal:** know what each release file contributes.

**Why:** SUB finds filings and periods, TAG maps standard XBRL tags, NUM contains facts such as the filed total, and SOI contains the long-format schedule.

**What could go wrong:** NUM is large; reading all of it into an unfiltered DataFrame wastes memory. SUB also excludes superseded rows by `prevrpt`.

In [5]:
def data_rows(path):
    with Path(path).open('rb') as handle:
        return max(sum(1 for _ in handle) - 1, 0)

counts = pd.DataFrame([
    ('SUB', len(sub), 'filings and periods'),
    ('TAG', len(tag), 'standard and custom tag labels'),
    ('NUM', data_rows(release_dir / 'datasets' / 'num.tsv'), 'numeric facts, including filed totals'),
    ('SOI', data_rows(release_dir / 'soi.tsv'), 'schedule-of-investments rows'),
], columns=['file', 'rows loaded/reported', 'what it contributes'])
display(counts)

,file,rows loaded/reported,what it contributes
0,SUB,182,filings and periods
1,TAG,44302,standard and custom tag labels
2,NUM,1011638,"numeric facts, including filed totals"
3,SOI,197160,schedule-of-investments rows


## Step 3 — The naive sum and the filed total

**Goal:** compare a direct current-period SOI sum with the BDC's own reported `InvestmentOwnedAtFairValue`.

**Why:** category rows, repeated issuer tables and dimension handling can make a raw sum differ from the total printed by the filer. A reconciliation gap reveals work to do; it is not a hidden adjustment.

**What could go wrong:** summing both comparative and current dates double-counts valuations, and the filed total is a NUM fact rather than a sum of the human-readable page.

In [6]:
canonical = clean.recover_axis_identifiers(
    holdings.split_periods(canonical_base.copy(), sub)
)
coalesced_all_dates = coalesce.coalesce(canonical)
candidates = coalesced_all_dates[coalesced_all_dates['is_current']].copy()
raw_current = canonical[canonical['is_current']]
num = load.load_num(
    release_dir,
    tags=[reconcile.FV_TAG],
    adsh={ADSH},
    custom_fv=True,
)
filed_trace = reconcile.filed_totals(num, sub[sub['adsh'].eq(ADSH)]).iloc[0]
naive_total = raw_current['fair_value'].sum()
coalesced_total = candidates['fair_value'].sum()
filed_total = filed_trace['filed_fv']

pd.DataFrame([
    ('raw current SOI fair-value facts', naive_total),
    ('coalesced candidate holdings', coalesced_total),
    ('undimensioned filed fair value', filed_total),
], columns=['stage', 'fair value']).assign(
    relative_gap=lambda d: d['fair value'] / filed_total - 1
).style.format({'fair value': '${:,.0f}', 'relative_gap': '{:+.3%}'})

,stage,fair value,relative_gap
0,raw current SOI fair-value facts,"$9,612,393,921",+892.763%
1,coalesced candidate holdings,"$973,328,308",+0.525%
2,undimensioned filed fair value,"$968,246,417",+0.000%


**Expected output:** current SOI facts and coalesced candidates are close because this filing stores each investment fair value once. The filed total still differs, so later structural rules and the tolerance decide admission.

**What could go wrong:** missing fair values are not zero. Keep them missing until a documented transformation or exclusion establishes otherwise.

## Step 4 — Map by XBRL tag and coalesce into holdings

**Goal:** turn long facts into one candidate per filing, date and investment identifier.

**Why:** the standard tags are stable across filer layouts. Coalescing combines values and dimensions while preserving every source row for Priya's audit.

**What could go wrong:** coalescing by display text would be fragile. The identifier can contain issuer and security information and should not be shortened by regex guesswork.

In [7]:
trace_holding = candidates[candidates['identifier'].eq(TRACE_IDENTIFIER)].iloc[0]
pd.DataFrame([trace_holding[[
    'identifier', 'fair_value', 'cost', 'shares', 'pct_net_assets',
    'source_rows', 'n_rows', 'n_conflicts'
]]])

,identifier,fair_value,cost,shares,pct_net_assets,source_rows,n_rows,n_conflicts
1,2X LLC Investors LP Class A Units,1220895.0,176915.0,43875.0,0.0033,42670;42671,2,0


**Expected output:** one holding with source rows `42670;42671`, two source rows and zero conflicts. `n_rows > 1` is expected: XBRL placed values and dimensions on different rows.

**What could go wrong:** `n_rows` is not the number of BDC investments. It is the number of SOI facts coalesced into one investment.

## Step 5 — Each transformation and cleaning rule

**Goal:** audit every change in order and attach a real example, row count and traced-filing before/after.

**Why:** a rule should be explainable from structure. The release-wide coverage in the cleaning log was measured after rule selection and must not be used to reverse-engineer a rule.

**What could go wrong:** changing rule order changes candidates. Apply the published order exactly. `transform_cleaning_attributes` classifies fields but does not delete rows.

In [8]:
log = pd.read_csv(root / 'reports' / 'cleaning_log.csv')
log.set_index('rule')[['artifact', 'example_adsh', 'example_soi_row',
                       'rows_affected', 'coverage_before', 'coverage_after']]

,artifact,example_adsh,example_soi_row,rows_affected,coverage_before,coverage_after
rule,,,,,,
transform_current_period,comparative and other-date rows excluded from ...,0000017313-26-000095,42732,64835,0.285,0.285
transform_axis_identifiers,identifier recovered from a fallback axis for ...,0001493152-26-036285,10586,39,0.285,0.285
transform_custom_fair_values,fair value recovered from a custom NUM tag key...,0001143513-26-000012,75371,28,0.285,0.271
transform_cleaning_attributes,"equity/warrant security class, affiliate categ...",0000017313-26-000095,42733,68175,0.271,0.271
rule_totals,identifier is a total/subtotal or category-onl...,0000930413-26-002499,106977,59,0.271,0.288
rule_issuer_aggregates,short issuer summary duplicates a detailed row,0000017313-26-000095,40610,311,0.288,0.400
rule_cash,cash or money-market line in schedule,0001104659-26-094146,81264,40,0.400,0.421


In [9]:
def snapshot(frame):
    value = frame['fair_value'].sum()
    return len(frame), (None if pd.isna(value) else float(value))

trace = []
def add_trace(step, before, after):
    before_n, before_fv = snapshot(before)
    after_n, after_fv = snapshot(after)
    trace.append({
        'step': step,
        'holdings before': before_n,
        'holdings after': after_n,
        'fair value before': before_fv,
        'fair value after': after_fv,
        'traced rows changed': before_n - after_n,
    })

pre_axis = holdings.split_periods(canonical_base.copy(), sub)
pre_axis_all = coalesce.coalesce(pre_axis)
axis_restored = clean.recover_axis_identifiers(pre_axis)
axis_all = coalesce.coalesce(axis_restored)
add_trace('transform_axis_identifiers', pre_axis_all, axis_all)
add_trace('transform_current_period', axis_all,
          axis_all[axis_all['is_current']])

current = axis_all[axis_all['is_current']].copy()
recovered = clean.recover_custom_fair_values(current, num, sub)
add_trace('transform_custom_fair_values', current, recovered)

classified = clean.add_cleaning_attributes(recovered)
add_trace('transform_cleaning_attributes', recovered, classified)

working = classified
for rule in clean.CLEANING_RULES:
    drop, _ = rule(working)
    after = working.loc[~drop].copy()
    add_trace(rule.__name__, working, after)
    working = after

display(pd.DataFrame(trace).style.format({
    'fair value before': '${:,.0f}',
    'fair value after': '${:,.0f}',
}))
print('Final traced holdings:', len(working))

,step,holdings before,holdings after,fair value before,fair value after,traced rows changed
0,transform_axis_identifiers,808,808,"$2,003,084,769","$2,003,084,769",0
1,transform_current_period,808,403,"$2,003,084,769","$973,328,308",405
2,transform_custom_fair_values,403,403,"$973,328,308","$973,328,308",0
3,transform_cleaning_attributes,403,403,"$973,328,308","$973,328,308",0
4,rule_totals,403,403,"$973,328,308","$973,328,308",0
5,rule_issuer_aggregates,403,402,"$973,328,308","$971,696,643",1
6,rule_cash,402,402,"$971,696,643","$971,696,643",0


Final traced holdings: 402


**Expected output:** the issuer-aggregate rule removes the short Stellus `J.R. Watkins, LLC Term Loan` summary from source rows 122368–122369. The final panel has 402 holdings. The other two structural rules remove nothing from this traced filing.

**Why several steps fire but few rows change:** period selection, coalescing and attribute classification are transformations. Ordinary holdings do not need to be deleted for a filing to pass through the method.

## Step 6 — Reconcile, admit and measure coverage

**Goal:** calculate the filing gap using the pipeline's tolerance and distinguish filing coverage from dollar coverage.

**Why:** Tom needs to know what share of the release a number represents, while Priya needs the filing's reported total beside the modelled holdings.

**What could go wrong:** 88 admitted filings is not 88% of anything in dollar terms. Both denominators are required.

In [10]:
tolerance = float(params['reconcile']['tolerance'])
reconciled = reconcile.reconcile(
    working, filed_trace.to_frame().T,
    sub[sub['adsh'].eq(ADSH)], tolerance, soi_adsh={ADSH}
)
display(reconciled[['adsh', 'name', 'n_holdings', 'holdings_fv',
                    'filed_fv', 'gap_pct', 'admitted']].style.format({
    'holdings_fv': '${:,.0f}', 'filed_fv': '${:,.0f}', 'gap_pct': '{:+.3%}'
}))
coverage = json.loads((root / 'reports' / 'coverage.json').read_text())
print('Tolerance:', f'{tolerance:.2%}')
print('Coverage:', json.dumps(coverage, indent=2))

,adsh,name,n_holdings,holdings_fv,filed_fv,gap_pct,admitted
0,0001104659-26-093435,STELLUS CAPITAL INVESTMENT CORP,402,"$971,696,643","$968,246,417",+0.356%,True


Tolerance: 1.00%
Coverage: {
  "filings_attempted": 177,
  "filings_admitted": 88,
  "share_of_filings": 0.497,
  "filed_fv_attempted": 556908483659.0,
  "filed_fv_admitted": 234366032543.0,
  "share_of_dollars": 0.421,
  "filings_without_soi_rows": 5
}


**Expected output:** Stellus is admitted. Its holdings total and filed total are not identical, but the absolute gap is below one percentage point. Release coverage remains **49.7% of filings** and **42.1% of filed dollars**.

**What could go wrong:** do not label the reconciled subset complete. Five filings without SOI rows are separate from the 177 attempted filings.

## Step 7 — Inspect an exclusion instead of forcing it

**Goal:** see the evidence for a filing outside the panel.

**Why:** changing a rule until an excluded target passes leaks the desired answer into the method. Structural evidence must support the decision even when it leaves a gap.

**What could go wrong:** a custom identifier column can mix category and issuer members; its presence does not make it a safe investment identifier.

In [11]:
coverage_rows = pd.read_csv(root / 'reports' / 'coverage.csv', dtype={'adsh': str})
partners = coverage_rows[coverage_rows['name'].str.contains('PARTNERS GROUP', na=False)].iloc[0]
triage = pd.read_csv(root / 'docs' / 'triage_notes.csv', dtype=str, keep_default_na=False)
partners_note = triage[triage['adsh'].eq(partners['adsh'])].iloc[0]
pd.DataFrame({
    'field': ['filing', 'category', 'gap', 'evidence', 'example', 'fix'],
    'value': [
        partners['name'], partners_note['triage_category'],
        'n/a' if pd.isna(partners['gap_pct']) else f"{partners['gap_pct']:+.2%}",
        partners_note['evidence'], partners_note['example'], partners_note['fix'],
    ],
})

,field,value
0,filing,"PARTNERS GROUP LENDING FUND, LLC"
1,category,no_identified_holdings
2,gap,n/a
3,evidence,"457 current-period SOI rows, none with 'Invest..."
4,example,0001398344-26-014697 / soi_row 14549 (current-...
5,fix,None possible from SOI: no SOI row of this fil...


**Expected output:** Partners Group Lending Fund has no identified holdings from the standard SOI identifier and is categorized `no_identified_holdings`. Its custom `InvestmentsIdentifier` field mixes dimensions, so the conservative choice is to retain the exclusion.

**What could go wrong:** excluding a filing does not mean it invested nothing. The app keeps it visible with evidence.

## Step 8 — Verify against the filing and name the total's source

**Goal:** compare sampled values with the human-readable schedule and state what “filed total” means.

**Why:** Priya needs to verify that the extraction did not invent or silently alter values. The filing total has its own provenance in NUM.

**What could go wrong:** a human-readable page can round values or omit words; a match is spot-check evidence, not proof of every interpretation.

In [12]:
panel_example = working[working['identifier'].eq(TRACE_IDENTIFIER)].iloc[0]
filing_fact = num[
    num['tag'].eq(reconcile.FV_TAG) & num['dimn'].eq('0') &
    num['ddate'].eq(PERIOD)
]
if SEC_AVAILABLE:
    comparison = pd.DataFrame([
        ('security', cells[0], panel_example['identifier']),
        ('fair value', cells[28], f"{panel_example['fair_value']:,.0f}"),
        ('amortized cost', cells[25], f"{panel_example['cost']:,.0f}"),
        ('shares', cells[22], f"{panel_example['shares']:,.0f}"),
        ('percent of net assets', cells[30] + cells[31],
         f"{panel_example['pct_net_assets']:.2%}"),
    ], columns=['field', 'filing display', 'LOOM panel'])
    display(comparison)
else:
    print('Live filing comparison skipped because the SEC fetch failed.')

print('Filed total source: undimensioned NUM fact', reconcile.FV_TAG)
display(filing_fact[['tag', 'dimn', 'ddate', 'value']].reset_index(drop=True))

,field,filing display,LOOM panel
0,security,2X Investors LP Class A Units,2X LLC Investors LP Class A Units
1,fair value,"1,220,895","1,220,895"
2,amortized cost,"176,915","176,915"
3,shares,"43,875","43,875"
4,percent of net assets,0.33%,0.33%


Filed total source: undimensioned NUM fact InvestmentOwnedAtFairValue


,tag,dimn,ddate,value
0,InvestmentOwnedAtFairValue,0,2026-06-30,968246417.0


**Expected output:** the 2X equity security's shares, cost and fair value agree with the displayed filing table. The filed **$968,246,417** total comes from the undimensioned `InvestmentOwnedAtFairValue` NUM fact at 2026-06-30, not from summing the filing's rendered HTML.

**What could go wrong:** if SEC retrieval fails, do not fabricate a comparison. Use the release rows and state that the live check was skipped.

## Step 9 — Reproduce the result

**Goal:** rebuild and verify from tracked inputs.

**Why:** a saved notebook is not enough; the classmate must be able to rerun the pipeline.

Run from the repository root:

```bash
dvc pull
dvc repro
PYTHONPATH=src python -m lookthrough.check
pytest -q
```

Then build and test the app:

```bash
python scripts/export_app_data.py
pytest -q tests/test_app.py
```

And run it locally:

```bash
PYTHONPATH=. .venv/bin/uvicorn app.api.main:app --reload --port 8000
.venv/bin/python -m http.server 8001 --directory app/web
```

**Expected output:** DVC identifies the same stages, the self-check passes all 10 checks, and pytest passes. The exporter asserts that copied totals equal `reports/coverage.json` and each admitted panel sum.

**What could go wrong:** a fresh clone needs `dvc pull` for release/output inputs and `export_app_data.py` for the app's generated JSON. Neither app nor lab code is a DVC stage.

## Step 10 — What this does not do

This is an auditable investment-holdings view, not a cross-company entity resolver or credit model.

- No borrower matching across BDCs or periods.
- No inferred credit risk, default probability or return signal.
- No filling of unreconciled schedules.
- No claim that filing coverage equals dollar coverage.
- No write path: every app API endpoint is GET-only.

**The conservative stopping point:** when the schedule structure cannot support an investment identity or a filing cannot be reconciled within tolerance, retain the evidence and exclude the filing.